In [6]:
import time
import RemoteScreen
from PIL import Image
import os
import numpy as np
import zwoasi as asi
import matplotlib.pyplot as plt
from pathlib import Path

In [7]:

def TakePicture():
    try:
        camera_info = camera.get_camera_property()
        print(f"Connected to: {camera_info['Name']}")

        # 4. Basic controls (exposure is in microseconds)
        FUCK =   0.25 # in seconds (just use input)
        SHGIT = FUCK * 1000000
        BISH = int(SHGIT)
        camera.set_image_type(asi.ASI_IMG_RAW16)          # 16-bit mono
        camera.set_control_value(asi.ASI_GAIN, 100)
        camera.set_control_value(asi.ASI_EXPOSURE, BISH)  # 1 s
        camera.set_control_value(asi.ASI_BANDWIDTHOVERLOAD, 40)

        # 5. Capture
        print("Capturing image...")
        camera.start_exposure(is_dark=False)

        timeout = time.time() + 30  # exposure time + margin
        while True:
            status = camera.get_exposure_status()
            if status == asi.ASI_EXP_SUCCESS:
                break
            if status == asi.ASI_EXP_FAILED:
                raise RuntimeError("Exposure failed!")
            if time.time() > timeout:
                raise TimeoutError("Exposure timed out")
            time.sleep(0.05)

        # 6. Download: the SDK returns a raw bytearray, so convert it to a NumPy array
        raw = camera.get_data_after_exposure()
        width, height, binning, img_type = camera.get_roi_format()
        dtype = np.uint16 if img_type == asi.ASI_IMG_RAW16 else np.uint8
        image_data = np.frombuffer(raw, dtype=dtype).reshape(height, width)
        MakeImage(image_data)

        print(f"Captured image shape: {image_data.shape}, dtype: {image_data.dtype}")
        print(f"min={image_data.min()}, max={image_data.max()}, mean={image_data.mean():.1f}")

    finally:
        # Always release the camera, even if something above failed
        camera.close()


In [8]:
# 1. Load the ZWO SDK library.
    # Set the ZWO_ASI_LIB environment variable, or edit the fallback path
    # (.dll on Windows, .so on Linux, .dylib on macOS).


sdk_path = os.environ.get("ZWO_ASI_LIB", r"C:\Program Files\ASIStudio\ASICamera2.dll")
try:
    asi.init(sdk_path)
except Exception as e:
        # Re-running the cell in the same kernel re-inits the SDK; that's harmless.
    if "already" not in str(e).lower():
        raise

    # 2. Check connected cameras
num_cameras = asi.get_num_cameras()
if num_cameras == 0:
    raise RuntimeError("No ZWO cameras found!")  # exit() would kill the notebook kernel

print(f"Found {num_cameras} camera(s).")
for idx, name in enumerate(asi.list_cameras()):
    print(f"[{idx}] {name}")

    # 3. Open the camera (index 0)
camera = asi.Camera(0)

Found 1 camera(s).
[0] ZWO ASI6200MM Pro


In [9]:
def MakeImage(image_data):
    file_name = str(input("Enter image name: "))
    # 1. Lossless 16-bit PNG: keeps all the data, but looks very dark in most viewers
    Image.fromarray(image_data).save("capture_16bit.png")
    print("1")

    # 2. Stretched 8-bit version for viewing (JPEG can only hold 8 bits per pixel)
    lo, hi = np.percentile(image_data, (0.5, 99.5))   # ignore extreme outliers
    hi = max(hi, lo + 1)                              # avoid divide-by-zero on flat frames
    img8 = (np.clip((image_data.astype(np.float32) - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    directory = Path(r"C:\Berkeley\Mike Group\Scene Generator\Camera Photos")
    full_path = directory / file_name
    full_path = full_path.with_suffix(".png")
    Image.fromarray(img8).save(full_path)
    print("2")

    # 3. Show it inline in the notebook
    plt.figure(figsize=(10, 7))
    plt.imshow(img8, cmap="gray")
    plt.axis("off")
    plt.show()
    print("3")

    print("Saved to:", full_path)


In [10]:
"""
*   Leo Neat
*   Jet Propulsion Laboratory
*   Division 38 Optics
*   leo.s.neat@jpl.nasa.gov
*   I did things too
*   -Forrest Johnson
*   LowFluxClient.py
*       This class is essentially the text based user interface for the RemoteScreen class. This class makes it so
*   all of the required commands are called in the right order and It should be used as a guide to how the Remote Screen
*   class functions.
*
"""
# Source - https://stackoverflow.com/a/21946788
# Posted by David Ketcheson
# Retrieved 2026-09-10, License - CC BY-SA 3.0
22



raw_input = input


def display_menu():
    # Function to display all of the options the user has when running this program
    # Note: Correct input is assumed

    print("")
    print("")
    print("_______________________")
    print("*Low Flux Android Menu*")
    print("_______________________")
    print("1) Clear Screen")
    print("2) Set flat field")
    print("3) Set a Matrix")
    print("4) Set an Image")
    print("5) Disconnect")
    print("6) Set Time Delay")
    print("7) View Current Display")
    print("8) Set Screen Brightness")
    result = int(raw_input("Enter your choice: "))
    return result


def head():
    #   This is the main body for the LowFlux client script. This is where all of the decided is done and is just a
    # lot of if statements that correspond to options that are performable by the remote screen.

    screen.print_state()
    result = display_menu()
    if result == 1:
        screen.set_off()
        screen.send_screen()
        
        head()
    elif result == 2:
        val = int(raw_input("What intensity do you want the screen(0-255): "))
        depth = int(raw_input("What depth do you want to put the field(0=R, 1=G, 2=B): "))
        screen.set_off()
        screen.set_flatfield(val, depth)
        screen.send_screen()
        TakePicture()
        while True:
            Again = input("Would you like to take another image with the same settings (Y/N)?: ")
            if Again  == "Y":
                TakePicture()
            else:
                break
        head()
    elif result == 3:
        ret = matmenu()
        if ret  == 1:
            rad = int(raw_input("Enter the radius: "))
            bval = int(raw_input("Enter the circle pixel values[0-255]: "))
            dval = int(raw_input("Enter the outer pixel values[0-255]: "))
            cx = int(raw_input("Enter the center x pixel: "))
            cy = int(raw_input("Enter the center y pixel: "))
            depth = int(raw_input("What depth do you want to put the field(0=R, 1=G, 2=B): "))
            screen.set_circle(depth, rad, bval, dval, cx, cy)
            screen.send_screen()
            TakePicture()
            while True:
                        Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                        if Again  == "Y":
                            TakePicture()
                        else:
                            break
        elif ret == 2:
            xspace = int(raw_input("Enter the distance between the pixels x values: "))
            yspace = int(raw_input("Enter the distance between the pixels y values: "))
            bval   = int(raw_input("Enter the selected pixels value[0-255]: "))
            dval   = int(raw_input("Enter the non selected pixels value[0-255]: "))
            depth = int(raw_input("What depth do you want to put the field(0=R, 1=G, 2=B): "))
            screen.set_sparsefield(depth, xspace, yspace, bval, dval)
            screen.send_screen()
            TakePicture()
            while True:
                        Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                        if Again  == "Y":
                            TakePicture()
                        else:
                            break
        elif ret == 3:
            px = int(raw_input("Enter the x value of the pixel you want to change: "))
            py = int(raw_input("Enter the y value of the pixel you want to change: "))
            pz = int(raw_input("Enter the z value of the pixel you want to change: "))
            inten = int(raw_input("Enter the intensity of the pixel you want to change: "))
            screen.set_pixel(px, py, pz, inten)
            screen.send_screen()
            TakePicture()
            while True:
                        Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                        if Again  == "Y":
                            TakePicture()
                        else:
                            break
        else:
            print("invalid opt, returning to main screen")
        head()
    elif result == 4:
        screen.set_image()
        screen.send_screen()
        TakePicture()
        while True:
                    Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                    if Again  == "Y":
                        TakePicture()
                    else:
                        break
        head()
    elif result == 5:
        screen.disconnect()
    elif result == 6:
        newtime = int(raw_input("Please Enter the Delay you want in ms(0 is always on): "))
        newsleep = int(raw_input("Please Enter the length of time you want the image to say on(ms): "))
        screen.set_time(newtime, newsleep)
        TakePicture()
        while True:
                    Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                    if Again  == "Y":
                        TakePicture()
                    else:
                        break
        head()
    elif result == 7:
        img = Image.open("img.bmp")
        img.show()
        head()
    elif result == 8:
        brightness = int(raw_input("Please enter the new screen brighness(0-255): "))
        screen.set_phone_brightness(brightness)
        TakePicture()
        while True:
                    Again = input("Would you like to take another image with the same settings (Y/N)?: ")
                    if Again  == "Y":
                        TakePicture()
                    else:
                        break
        head()
    else:
        print("invalid option please repick...")
        head()


def matmenu():
    # This is the menu that is printed when the user decides that they want to use a matrix as input

    print("1) Send a Circle")
    print("2) Send a Grid")
    print("3) Set an individual Pixel")
    ret = int(raw_input("Enter your choice:"))
    return ret

# Global Screen what acts as the main data type
# Note that the screen resolution is hardcoded
try:
    screen = RemoteScreen.RemoteScreen()
    head()                                              # Prompt menus to open
except:
    screen.disconnect()
    raise


Error unable to establish connection: [WinError 10061] No connection could be made because the target machine actively refused it


NameError: name 'screen' is not defined